# SignalShap — Full Study on Apple Silicon (M4, 48 GB)

Runs the complete study at a scale the CI sandbox could not reach. The sandbox was capped at
**3 GB RAM / 1,500 users**; here we auto-size each corpus to a memory budget you control.

**What this notebook produces**
1. Four real corpora (MovieLens-1M + the three LightGCN benchmark splits)
2. E0 preconditions → exact Shapley → LOO comparison
3. **E9 redundancy intervention** — the paper's central claim, with ground truth by construction
4. Full experiment suite on every corpus (robustness, segments, fusion, LightGCN/SASRec)
5. Regenerated `paper/` figures and tables

**Expect ~45–90 min** for the full run. Every step is checkpointed to `artefacts/`, so you
can interrupt and resume.

> **Why memory dominates.** The game holds five dense `n_users × n_items` float32 score
> matrices simultaneously. Amazon-Book at full size is **96 GB** — beyond 48 GB — so
> subsampling is unavoidable there and is disclosed in the paper rather than hidden.

## 1 · Environment

Apple Silicon note: install NumPy/SciPy via `pip` wheels, which link against **Accelerate** and are already well optimised for M-series. No CUDA anywhere — the entire pipeline is CPU-only by design.

In [1]:
import platform, os, sys, subprocess
print("python :", platform.python_version())
print("machine:", platform.machine(), "|", platform.platform())
if platform.machine() != "arm64":
    print("  note: not arm64 — you may be under Rosetta; a native arm64 Python is faster.")

# Physical RAM (macOS)
try:
    ram = int(subprocess.check_output(["sysctl","-n","hw.memsize"]).strip())/1e9
    print(f"RAM    : {ram:.0f} GB")
except Exception:
    ram = 48.0
    print("RAM    : assuming 48 GB")

python : 3.12.13
machine: arm64 | macOS-26.5.1-arm64-arm-64bit
RAM    : 52 GB


In [2]:
%pip install -q numpy pandas scipy scikit-learn matplotlib pyyaml pyarrow tabulate jinja2 nbformat
import numpy as np, pandas as pd
print("numpy", np.__version__, "| pandas", pd.__version__)
np.show_config()  # confirm Accelerate / OpenBLAS backing


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
numpy 2.5.1 | pandas 3.0.5
Build Dependencies:
  blas:
    detection method: system
    found: true
    include directory: unknown
    lib directory: unknown
    name: accelerate
    openblas configuration: unknown
    pc file directory: unknown
    version: unknown
  lapack:
    detection method: system
    found: true
    include directory: unknown
    lib directory: unknown
    name: accelerate
    openblas configuration: unknown
    pc file directory: unknown
    version: unknown
Compilers:
  c:
    commands: cc
    linker: ld64
    name: clang
    version: 15.0.0
  c++:
    commands: c++
    linker: ld64
    name: clang
    version: 15.0.0
  cython:
    commands: cython
    linker: cython
    name: cython
    version: 3.2.8
Machine Information:
  build:
    cpu: aarch64
    endian: little
    family: aarch6

### Repo and data

Clone the repo, then fetch corpora. MovieLens-1M is already committed; the three LightGCN
splits come from `scripts/fetch_benchmarks.sh` (~37 MB).

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO = Path.home()/"signalshap-code"      # <-- edit if you cloned elsewhere
if not REPO.exists():
    subprocess.run(["git","clone",
        "https://github.com/mouadlouhichi/signalshap-code.git", str(REPO)], check=True)
os.chdir(REPO); sys.path.insert(0, str(REPO/"src"))
print("cwd:", Path.cwd())

# Benchmark corpora (idempotent)
if not (REPO/"data"/"raw"/"gowalla").exists():
    subprocess.run(["bash","scripts/fetch_benchmarks.sh"], check=True)

# MovieLens-1M lives at data/ml-1m/ ; the loader searches data/raw/, data/, and root
for p in ["data/ml-1m","data/raw/gowalla","data/raw/yelp2018","data/raw/amazon-book"]:
    print(("  OK   " if Path(p).exists() else "  MISS "), p)

cwd: /Users/mlouhichi


bash: scripts/fetch_benchmarks.sh: No such file or directory


CalledProcessError: Command '['bash', 'scripts/fetch_benchmarks.sh']' returned non-zero exit status 127.

## 2 · Memory budget and auto-sizing

Set `SCORE_BUDGET_GB` to how much RAM the five score matrices may occupy. Users per corpus
are then derived, not guessed:

$$\text{max\_users} = \frac{\text{budget}}{5 \times n_{items} \times 4\text{ bytes}}$$

On 48 GB, **16 GB is a safe default** — it leaves room for the coalition machinery, the ridge
fits, and the OS. Raise it if you have headroom; lower it if you hit swap.

In [5]:
SCORE_BUDGET_GB = 16.0        # <-- the one knob that matters
SEEDS           = (42, 43, 44)
CORPORA         = ["ml_1m", "yelp2018", "gowalla", "amazon_book_lgcn"]

ITEMS = {"ml_1m": 3533, "gowalla": 40981, "yelp2018": 38048, "amazon_book_lgcn": 91599}
FULL  = {"ml_1m": 6038, "gowalla": 29858, "yelp2018": 31668, "amazon_book_lgcn": 52643}

plan = {}
for k in CORPORA:
    cap = int(SCORE_BUDGET_GB*1e9 / (5*ITEMS[k]*4))
    plan[k] = min(cap, FULL[k])
    pct = 100*plan[k]/FULL[k]
    print(f"  {k:18s} items={ITEMS[k]:6d}  users={plan[k]:6d}/{FULL[k]:6d} "
          f"({pct:5.1f}%)  {'FULL' if plan[k]>=FULL[k] else 'subsampled'}")
print(f"\nSandbox ran 1,500 users/corpus at 3 GB — this is a large scale-up.")

  ml_1m              items=  3533  users=  6038/  6038 (100.0%)  FULL
  yelp2018           items= 38048  users= 21026/ 31668 ( 66.4%)  subsampled
  gowalla            items= 40981  users= 19521/ 29858 ( 65.4%)  subsampled
  amazon_book_lgcn   items= 91599  users=  8733/ 52643 ( 16.6%)  subsampled

Sandbox ran 1,500 users/corpus at 3 GB — this is a large scale-up.


## 3 · Pre-registered configuration

`configs/frozen.yaml` holds values fixed **before** results were seen: per-corpus `N_max`,
the frozen ridge `λ`, and the `v₀` permutation seed. `λ` is *never* tuned per coalition —
doing so is optimistic bias that grows with `|S|`.

In [6]:
from signalshap.config import FrozenConfig, SOURCES, write_artefact, read_artefact

cfg = FrozenConfig.load()
print("sources:", SOURCES)
print("N_max  :", cfg.n_max)
print("lambda :", cfg.ridge_lambda, "(frozen)")
print("gate   : candidate recall >=", cfg.recall_gate)

sources: ('cf', 'ct', 'pop', 'rec', 'seq')
N_max  : {'ml_1m': 600, 'lastfm_2k': 500, 'amazon_book': 1000, 'gowalla': 2500, 'yelp2018': 5000, 'amazon_book_lgcn': 5000}
lambda : 1.0 (frozen)
gate   : candidate recall >= 0.6


## 4 · Load corpora and check the density ordering

Density is **computed as-used**, never quoted from a source paper — the subsample alone
moves it. The four corpora should span roughly 25×.

⚠️ The LightGCN splits carry **no timestamps**, so leave-last-out *temporal* splitting applies
to MovieLens only. On the others, interaction order is arbitrary, which structurally
handicaps the `rec` and `seq` players — their attributions there are **lower bounds**.

In [7]:
import warnings, gc
from signalshap.data.loaders import load_dataset, build_dataset_stats

datasets, rows = {}, []
for k in CORPORA:
    os.environ["SIGNALSHAP_MAX_USERS"] = str(plan[k])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ds = load_dataset(k)
    datasets[k] = ds
    s = ds.stats()
    rows.append({"corpus": k, "users": s["users"], "items": s["items"],
                 "interactions": s["interactions"],
                 "density %": round(s["density"]*100, 4),
                 "temporal split": not s.get("no_timestamps", False),
                 "real": not s["synthetic"]})
display(pd.DataFrame(rows).sort_values("density %", ascending=False))

stats = build_dataset_stats(datasets)
write_artefact("dataset_stats.json", stats)
d = {k: stats[k]["density"] for k in CORPORA}
print(f"density range: {max(d.values())/min(d.values()):.0f}x "
      f"({min(d.values())*100:.4f}% .. {max(d.values())*100:.4f}%)")

[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
[signalshap] yelp2018: loaded REAL data (21,026 users x 38,048 items, 839,455 interactions)
[signalshap] gowalla: loaded REAL data (19,521 users x 40,981 items, 549,280 interactions)
[signalshap] amazon_book_lgcn: loaded REAL data (8,733 users x 86,537 items, 398,784 interactions)


,corpus,users,items,interactions,density %,temporal split,real
0,ml_1m,6038,3533,575276,2.6967,True,True
1,yelp2018,21026,38048,839455,0.1049,False,True
2,gowalla,19521,40981,549280,0.0687,False,True
3,amazon_book_lgcn,8733,86537,398784,0.0528,False,True


density range: 51x (0.0528% .. 2.6967%)


## 5 · E9 — the redundancy intervention (run this first)

**This is the paper's central claim, so it runs before anything else.**

Observational comparison can only show LOO and Shapley *differ*; it cannot say which is
right. Here we inject a known duplicate of a real source, holding candidates fixed, which
fixes the correct answer by construction:

- **Symmetry** → the duplicated pair must receive *equal* credit
- **Property 2** → LOO must *collapse to zero* for both

If exact Shapley fails this, the method is wrong and the experiment says so.

In [8]:
import time
from signalshap.pipeline import Experiment
from signalshap.experiments.recovery import recovery_experiment, summarise

recovery = {}
for k in CORPORA:
    os.environ["SIGNALSHAP_MAX_USERS"] = str(plan[k])
    t0 = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        exp = Experiment(k, cfg, seed=42, synthetic=False)
        rec = recovery_experiment(exp, "cf", etas=(0.0, 0.01, 0.1, 0.5), seed=42)
    recovery[k] = rec
    write_artefact(f"e9_recovery_{k}.json", rec)
    c = rec["conditions"]["eta=0.0"]
    print(f"{k:18s} {time.time()-t0:5.0f}s  symmetry_err={c['symmetry_abs_error']:.2e}  "
          f"LOO={c['loo_original']:.5f}/{c['loo_duplicate']:.5f}", flush=True)
    del exp; gc.collect()

[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
    [E9 ml_1m] eta=0.0 (1/4) -- 2^6=64 coalitions...
    [E9 ml_1m] eta=0.0 done in 24.5s  symmetry_err=0.00e+00
    [E9 ml_1m] eta=0.01 (2/4) -- 2^6=64 coalitions...
    [E9 ml_1m] eta=0.01 done in 24.6s  symmetry_err=3.03e-05
    [E9 ml_1m] eta=0.1 (3/4) -- 2^6=64 coalitions...
    [E9 ml_1m] eta=0.1 done in 24.5s  symmetry_err=4.12e-05
    [E9 ml_1m] eta=0.5 (4/4) -- 2^6=64 coalitions...
    [E9 ml_1m] eta=0.5 done in 24.5s  symmetry_err=4.45e-05
ml_1m                131s  symmetry_err=0.00e+00  LOO=0.00000/0.00000
[signalshap] yelp2018: loaded REAL data (21,026 users x 38,048 items, 839,455 interactions)
    [E9 yelp2018] eta=0.0 (1/4) -- 2^6=64 coalitions...
    [E9 yelp2018] eta=0.0 done in 803.1s  symmetry_err=0.00e+00
    [E9 yelp2018] eta=0.01 (2/4) -- 2^6=64 coalitions...
    [E9 yelp2018] eta=0.01 done in 1408.2s  symmetry_err=7.23e-06
    [E9 yelp2018] eta=0.1 (3/4) -- 2^6=64 coalitions..

In [9]:
summary = pd.DataFrame([{
    "corpus": k,
    "density %": round(stats[k]["density"]*100, 4),
    "phi_cf": round(r["conditions"]["eta=0.0"]["phi_original"], 5),
    "phi_cf_dup": round(r["conditions"]["eta=0.0"]["phi_duplicate"], 5),
    "symmetry err": f"{r['conditions']['eta=0.0']['symmetry_abs_error']:.1e}",
    "LOO(cf)": round(r["conditions"]["eta=0.0"]["loo_original"], 5),
    "LOO(dup)": round(r["conditions"]["eta=0.0"]["loo_duplicate"], 5),
} for k, r in recovery.items()])
display(summary)

ok = all(r["conditions"]["eta=0.0"]["symmetry_abs_error"] < 1e-6 for r in recovery.values())
print("Shapley recovers symmetry on every corpus:", ok)
print("LOO collapses to ~0 on every corpus     :",
      all(abs(r["conditions"]["eta=0.0"]["loo_original"]) < 1e-6 for r in recovery.values()))
print("\n-> An engineer running standard ablation on a system with two redundant")
print("   components is told NEITHER contributes, and would retire both.")

,corpus,density %,phi_cf,phi_cf_dup,symmetry err,LOO(cf),LOO(dup)
0,ml_1m,2.6967,0.01078,0.01078,0.0e+00,0.0,0.0
1,yelp2018,0.1049,0.00618,0.00618,0.0e+00,0.0,0.0
2,gowalla,0.0687,0.01429,0.01429,0.0e+00,0.0,0.0
3,amazon_book_lgcn,0.0528,0.00251,0.00251,0.0e+00,0.0,0.0


Shapley recovers symmetry on every corpus: True
LOO collapses to ~0 on every corpus     : True

-> An engineer running standard ablation on a system with two redundant
   components is told NEITHER contributes, and would retire both.


### Graceful degradation

As η grows the duplicate becomes a noisier copy, exact redundancy weakens, and LOO becomes non-zero but erratic in sign — the behaviour Lemma 1(i) describes for ε-redundancy.

In [2]:
for k, r in recovery.items():
    print(f"\n{k}"); print(summarise(r).split("\n", 2)[2])

NameError: name 'recovery' is not defined

## 6 · Full experiment suite

E0 preconditions, exact Shapley, LOO, segments, fusion, robustness, and the LightGCN /
SASRec baselines. Checkpointed per corpus.

Memory note: `run_all()` holds the score matrices *and* the robustness sweeps. If a corpus
OOMs, lower `SCORE_BUDGET_GB` and re-run — only the failed corpus repeats.

In [ ]:
from signalshap.pipeline import run_multi_seed

results = {}
for k in CORPORA:
    out = Path(f"artefacts/results_{k}.json")
    if out.exists() and os.environ.get("FORCE_RERUN") != "1":
        results[k] = read_artefact(out.name); print(f"{k:18s} cached"); continue
    os.environ["SIGNALSHAP_MAX_USERS"] = str(plan[k])
    t0 = time.time()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            exp = Experiment(k, cfg, seed=42, synthetic=False)
            r = exp.run_all()
            r["multi_seed"] = run_multi_seed(k, cfg, SEEDS, False)
            r["e9_recovery"] = recovery[k]
        write_artefact(f"results_{k}.json", r); results[k] = r
        a, b = r["e0a_candidates"], r["e0b_monotonicity"]
        print(f"{k:18s} {time.time()-t0:5.0f}s  recall={a['candidate_recall']:.3f} "
              f"({'PASS' if a['gate_passes'] else 'FAIL'})  monot={b['violations']}/80", flush=True)
        del exp; gc.collect()
    except MemoryError:
        print(f"{k:18s} OOM — lower SCORE_BUDGET_GB and re-run this cell")

[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
ml_1m                291s  recall=0.748 (PASS)  monot=26/80
[signalshap] yelp2018: loaded REAL data (21,026 users x 38,048 items, 839,455 interactions)


### Preconditions come first

Candidate recall is the **ceiling** on every ranking metric — a test item outside $C_u$ scores
zero by construction. The monotonicity column decides whether Property 2 applies at all;
where it fails, negative $\varphi_g$ are substantive findings, not numerical error.

In [ ]:
pre = pd.DataFrame([{
    "corpus": k,
    "density %": round(r["dataset_stats"]["density"]*100, 4),
    "N_max": r["e0a_candidates"]["n_max"],
    "recall": round(r["e0a_candidates"]["candidate_recall"], 3),
    "gate": "PASS" if r["e0a_candidates"]["gate_passes"] else "FAIL",
    "monot. viol": f"{r['e0b_monotonicity']['violations']}/80",
    "Prop2 applies": r["e0b_monotonicity"]["property2_applicable"],
    "efficiency err": f"{r['e1_source_share']['efficiency']['abs_error']:.1e}",
} for k, r in results.items()])
display(pre)
print("Any corpus below 0.60 -> spec §2.2 ladder: raise N_max first (rung 1).")

### LOO vs Shapley — where they disagree in **sign**

In [ ]:
rows = []
for k, r in results.items():
    e2 = r["e2_loo_vs_shapley"]
    for g in SOURCES:
        l, s_ = e2["loo"][g], e2["shapley"][g]
        rows.append({"corpus": k, "source": g, "LOO": round(l,5), "Shapley": round(s_,5),
                     "gap": round(s_-l,5), "SIGN FLIP": "yes" if l*s_ < 0 else ""})
df = pd.DataFrame(rows); display(df)
print("sign flips per corpus:")
print(df[df["SIGN FLIP"]=="yes"].groupby("corpus").size().to_string() or "  none")

### Fusion vs neural baselines

All methods evaluated **full-catalog**; SignalShap-Fuse scores items outside $C_u$ as $-\infty$ so its recall ceiling is a visible cost, not a hidden denominator advantage.

In [ ]:
for k, r in results.items():
    e4 = r.get("e4_signalshap_fuse")
    if not e4: print(f"{k}: fusion not run"); continue
    print(f"\n=== {k} ===")
    display(pd.DataFrame([{"method": m, "NDCG@10": round(v["ndcg_at_10"],5),
                           "Recall@20": round(v["recall_at_20"],5)}
        for m, v in sorted(e4["full_catalog"].items(), key=lambda x:-x[1]["ndcg_at_10"])]))
    h = e4["holm_bonferroni"]
    for b, t in e4["wilcoxon"].items():
        print(f"   vs {b:22s} d={t['mean_diff']:+.5f}  Holm p={h['corrected'][b]:.4f}  "
              f"d_z={t['d_z']:+.3f}")

## 7 · Paper assets

Regenerates F1–F7 and T1–T8 into `artefacts/`, which `paper/paper.tex` reads directly. No number is ever typed into the manuscript by hand.

In [ ]:
from signalshap.plots.assets import generate_all_assets, FIG, TAB
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    assets = generate_all_assets(results, stats)
for kk, v in assets["figures"].items(): print(f"  {kk}: {v}")
print("  tables:", sorted(p.stem for p in TAB.glob("T*.md")))

In [ ]:
from IPython.display import Image, display as disp
for f in ["F2_shapley_shares","F3_loo_vs_shapley","F4_redundancy_heatmap","F6_fuse_gain"]:
    p = FIG/f"{f}.png"
    if p.exists(): print(f); disp(Image(filename=str(p)))

## 8 · Verify invariants

The tests that keep the work honest: the counterexample pinning the monotonicity hypothesis, and the density-ordering guard (which re-arms once three or more corpora are present).

In [ ]:
r = subprocess.run([sys.executable,"-m","pytest","tests/","-q","--no-header"],
                   capture_output=True, text=True)
print(r.stdout[-1800:])

## 9 · Build the PDF

Needs a TeX distribution: `brew install --cask mactex-no-gui` (or BasicTeX + `tlmgr install
natbib booktabs helvetic`).

In [ ]:
tex = subprocess.run(["which","pdflatex"], capture_output=True, text=True).stdout.strip()
if tex:
    print(subprocess.run(["make","-C","paper"], capture_output=True, text=True).stdout[-800:])
    print("PDF:", (REPO/"paper"/"paper.pdf").exists())
else:
    print("pdflatex not found — install MacTeX, then run:  make -C paper")

## 10 · Summary

Written to `artefacts/`: `results_<corpus>.json`, `e9_recovery_<corpus>.json`,
`dataset_stats.json`, `figures/F1–F7.png`, `tables/T1–T8.{md,tex,csv}`.

**Read T2 before anything else** — candidate recall bounds every metric below it, and the
monotonicity column decides whether Property 2 applies.

### What to check before submitting
1. Every corpus clears the **0.60 recall gate**, or is reported conditional on its ceiling
2. The **intervention** holds on all four (symmetry < 1e-6, LOO ≈ 0)
3. **No synthetic results** leak into `paper/` — the loader warns loudly if raw files are missing
4. Protocol differences (no timestamps on three corpora; subsampling) stay disclosed in §Limitations

In [ ]:
print("ARTEFACTS")
for p in sorted(Path("artefacts").rglob("*")):
    if p.is_file() and p.suffix in {".json",".png",".tex"}:
        print(f"  {p}  ({p.stat().st_size/1024:.0f} KB)")

print("\nHEADLINE")
for k, r in results.items():
    c = recovery[k]["conditions"]["eta=0.0"]
    print(f"  {k:18s} density={r['dataset_stats']['density']*100:.4f}%  "
          f"recall={r['e0a_candidates']['candidate_recall']:.3f}  "
          f"symmetry_err={c['symmetry_abs_error']:.1e}  LOO={c['loo_original']:.5f}")
print("\n'Exact' = exact GIVEN THE FITTED v. No sampling error; not no error.")